In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt

In [3]:
cols = [
    "id",
    "cycle",
    "op1",
    "op2",
    "op3"
]

for i in range(1,22):
    cols.append(f"s{i}")

train = pd.read_csv(
    "/content/drive/MyDrive/AI Work/CMAPSSData/train_FD001.txt",
    sep=r"\s+",
    header=None
)

train = train.iloc[:,:26]

train.columns = cols

In [4]:
max_cycle = (
    train.groupby("id")["cycle"]
    .max()
    .reset_index()
)

max_cycle.columns = [
    "id",
    "max_cycle"
]

train = train.merge(
    max_cycle,
    on="id"
)

train["RUL"] = (
    train["max_cycle"]
    -
    train["cycle"]
)

In [5]:
MAX_RUL = 125

train["RUL"] = train["RUL"].clip(
    upper=MAX_RUL
)

In [6]:
sensors = [
    "s2",
    "s3",
    "s4",
    "s7",
    "s8",
    "s11",
    "s12",
    "s13",
    "s15",
    "s17",
    "s20",
    "s21"
]

In [7]:
scaler = MinMaxScaler()

train[sensors] = scaler.fit_transform(
    train[sensors]
)

In [8]:
SEQ_LEN = 30

In [9]:
X = []
y = []

for engine_id in train["id"].unique():

    engine = train[
        train["id"] == engine_id
    ]

    data = engine[sensors].values

    rul = engine["RUL"].values

    for i in range(
        len(engine) - SEQ_LEN
    ):

        X.append(
            data[i:i+SEQ_LEN]
        )

        y.append(
            rul[i+SEQ_LEN]
        )

X = np.array(X)
y = np.array(y)

In [10]:
train_engines = list(
    range(1,81)
)

test_engines = list(
    range(81,101)
)

In [11]:
train_df = train[
    train["id"].isin(
        train_engines
    )
]

test_df = train[
    train["id"].isin(
        test_engines
    )
]

In [12]:
X = torch.tensor(
    X,
    dtype=torch.float32
)

y = torch.tensor(
    y,
    dtype=torch.float32
).view(-1,1)

In [13]:
class PositionalEncoding(
    nn.Module
):

    def __init__(
        self,
        d_model,
        max_len=5000
    ):

        super().__init__()

        pe = torch.zeros(
            max_len,
            d_model
        )

        position = torch.arange(
            0,
            max_len
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            )
            *
            (
                -np.log(10000.0)
                /
                d_model
            )
        )

        pe[:,0::2] = torch.sin(
            position*div_term
        )

        pe[:,1::2] = torch.cos(
            position*div_term
        )

        self.register_buffer(
            "pe",
            pe.unsqueeze(0)
        )

    def forward(
        self,
        x
    ):
        return (
            x
            +
            self.pe[
                :,
                :x.size(1)
            ]
        )

In [14]:
class TelemetryTransformer(
    nn.Module
):

    def __init__(self):

        super().__init__()

        self.embedding = nn.Linear(
            len(sensors),
            64
        )

        self.positional = (
            PositionalEncoding(
                64
            )
        )

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=64,
                nhead=4,
                batch_first=True
            )
        )

        self.encoder = (
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=3
            )
        )

        self.regressor = nn.Sequential(
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Linear(32,1)
        )

    def forward(
        self,
        x
    ):

        x = self.embedding(x)

        x = self.positional(x)

        x = self.encoder(x)

        x = x[:,-1,:]

        return self.regressor(x)

In [15]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = (
    TelemetryTransformer()
    .to(device)
)

criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0001
)

In [ ]:
epochs = 30

for epoch in range(epochs):

    model.train()

    optimizer.zero_grad()

    pred = model(
        X.to(device)
    )

    loss = criterion(
        pred,
        y.to(device)
    )

    loss.backward()

    optimizer.step()

    print(
        epoch,
        loss.item()
    )

In [ ]:
model.eval()

with torch.no_grad():

    pred = model(
        X.to(device)
    )

pred = (
    pred.cpu()
    .numpy()
    .flatten()
)

actual = (
    y.cpu()
    .numpy()
    .flatten()
)

rmse = np.sqrt(
    mean_squared_error(
        actual,
        pred
    )
)

print(
    "RMSE:",
    rmse
)

In [ ]:
plt.figure(
    figsize=(12,6)
)

plt.plot(
    actual[:500],
    label="Actual"
)

plt.plot(
    pred[:500],
    label="Predicted"
)

plt.legend()

plt.show()

In [ ]:
torch.save(
    model.state_dict(),
    "telemetry_transformer.pth"
)